# PL10 - PLNEB

In [ ]:
!pip uninstall -y peft
!pip install transformers==4.38.0 accelerate==0.27.2

In [ ]:
import transformers, accelerate
transformers.__version__, accelerate.__version__

('4.38.0', '0.27.2')

In [ ]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)

(True, 'Tesla T4')

In [ ]:
!pip install transformers seqeval evaluate datasets

## Data Loading

In [ ]:
from datasets import load_dataset

dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [ ]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

## Data Pre-Processing

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [ ]:
inputs = tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [ ]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

In [ ]:
tokens = ["as", "aulas", "plneb", "são", "interessantes", "!"]
tokenizer(tokens, is_split_into_words=True)

{'input_ids': [101, 260, 6880, 2322, 514, 22295, 453, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokens = ["as", "aulas", "plneb", "são", "interessantes", "!"]
inputs = tokenizer(tokens, is_split_into_words=True)

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [ ]:
inputs.word_ids()

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [ ]:
len(tokens), len(new_tokens)

(6, 10)

In [ ]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None

    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)
        elif previous_word != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_word = word_id

    return new_labels


def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(row["tokens"], is_split_into_words=True, truncation=True, max_length=512)
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels
        res.append(inputs)
    return res


train_data = tokenize_dataset(dataset_raw["train"])
test_data = tokenize_dataset(dataset_raw["test"])
print(len(train_data), len(test_data))

3716 930


In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


# TPC10

In [ ]:
label_list = dataset_raw["train"].features[f"ner_tags"].feature.names
label_list

['O',
 'B-Data',
 'I-Data',
 'B-Local',
 'I-Local',
 'B-Organizacao',
 'I-Organizacao',
 'B-Pessoa',
 'I-Pessoa',
 'B-Profissao',
 'I-Profissao']

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
# EVALUATE

import evaluate

seqeval = evaluate.load("seqeval")

In [ ]:
import numpy as np


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
# TRAIN
id2label = {
    0: "O",
    1: "B-Data",
    2: "I-Data",
    3: "B-Local",
    4: "I-Local",
    5: "B-Organizacao",
    6: "I-Organizacao",
    7: "B-Pessoa",
    8: "I-Pessoa",
    9: "B-Profissao",
    10: "I-Profissao",
}

label2id = {
    "O": 0,
    "B-Data": 1,
    "I-Data": 2,
    "B-Local": 3,
    "I-Local": 4,
    "B-Organizacao": 5,
    "I-Organizacao": 6,
    "B-Pessoa": 7,
    "I-Pessoa": 8,
    "B-Profissao": 9,
    "I-Profissao": 10,
}

## Model Training

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(label_list),   # número de classes do dataset
    id2label=id2label,    # mapeamento id → label
    label2id=label2id     # mapeamento label → id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner-pt",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)


from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.066391,0.947450,0.970244,0.958711,0.984894
2,No log,0.074803,0.947498,0.971193,0.959200,0.985419
3,0.017300,0.076959,0.949442,0.968978,0.959110,0.985069


Checkpoint destination directory ./ner-pt/checkpoint-233 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./ner-pt/checkpoint-466 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./ner-pt/checkpoint-699 already exists and is non-empty. Saving will proceed but saved results may be invalid.


TrainOutput(global_step=699, training_loss=0.015628990015076293, metrics={'train_runtime': 306.7257, 'train_samples_per_second': 36.345, 'train_steps_per_second': 2.279, 'total_flos': 658641856754904.0, 'train_loss': 0.015628990015076293, 'epoch': 3.0})

##Inference

In [ ]:
#para ver onde está o modelo final (config.json) que se pretende carregar
!ls -R ./ner-pt

./ner-pt:
checkpoint-233	checkpoint-466	checkpoint-699	runs

./ner-pt/checkpoint-233:
config.json	   scheduler.pt		    trainer_state.json
model.safetensors  special_tokens_map.json  training_args.bin
optimizer.pt	   tokenizer_config.json    vocab.txt
rng_state.pth	   tokenizer.json

./ner-pt/checkpoint-466:
config.json	   scheduler.pt		    trainer_state.json
model.safetensors  special_tokens_map.json  training_args.bin
optimizer.pt	   tokenizer_config.json    vocab.txt
rng_state.pth	   tokenizer.json

./ner-pt/checkpoint-699:
config.json	   scheduler.pt		    trainer_state.json
model.safetensors  special_tokens_map.json  training_args.bin
optimizer.pt	   tokenizer_config.json    vocab.txt
rng_state.pth	   tokenizer.json

./ner-pt/runs:
May06_19-00-06_3aea86cc2d1e  May06_20-49-56_3aea86cc2d1e
May06_19-35-22_3aea86cc2d1e

./ner-pt/runs/May06_19-00-06_3aea86cc2d1e:
events.out.tfevents.1778094014.3aea86cc2d1e.852.0

./ner-pt/runs/May06_19-35-22_3aea86cc2d1e:
events.out.tfevents.1778096125.3

In [ ]:
from transformers import pipeline

classifier = pipeline("ner", model="./ner-pt/checkpoint-466", tokenizer="./ner-pt/checkpoint-699", aggregation_strategy="first")

# Run inference
text="O antigo primeiro-ministro Pedro Passos Coelho classificou como absurda a proposta do Chega para baixar a idade da reforma, criticou o atual Governo PSD/CDS-PP por demorar a apresentar resultados e reiterou que não pretende regressar à política."

classifier(text)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'Profissao',
  'score': np.float32(0.9598498),
  'word': 'primeiro',
  'start': 9,
  'end': 17},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.99086666),
  'word': 'Pedro Passos Coelho',
  'start': 27,
  'end': 46}]